# DBSCAN from Scratch 🔍

In this notebook, we implement **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)**, a powerful density-based clustering algorithm.

## 📖 Theoretical Background

DBSCAN groups together points that are closely packed together (points with many nearby neighbors), marking as outliers points that lie alone in low-density regions.

### 1. Parameters
- **Epsilon ($\epsilon$)**: The maximum distance between two samples for one to be considered as in the neighborhood of the other.
- **MinPts**: The number of samples in a neighborhood for a point to be considered a **core point**.

### 2. Point Classifications
- **Core Point**: A point with at least `MinPts` within its $\epsilon$-neighborhood.
- **Border Point**: A point within the $\epsilon$-neighborhood of a core point, but with fewer than `MinPts` of its own.
- **Noise (Outlier)**: A point that is neither a core point nor a border point.

### 3. The Algorithm
1. Pick an arbitrary unvisited point $p$.
2. If $p$'s $\epsilon$-neighborhood has $\geq$ `MinPts`, create a new cluster $C$ and assign $p$ to $C$. Then, iteratively expand $C$ by checking the neighborhoods of all points in $C$.
3. If $p$'s $\epsilon$-neighborhood has $<$ `MinPts`, label $p$ as noise (can be reassigned later if found within another point's neighborhood).
4. Repeat until all points are visited.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

np.random.seed(42)

## 🏗️ The Implementation

In [ ]:
class DBSCAN:
    def __init__(self, eps=0.5, min_samples=5):
        self.eps = eps
        self.min_samples = min_samples
        self.labels_ = None
        
    def fit_predict(self, X):
        self.n_samples = X.shape[0]
        # Initialize all labels to unassigned (-2) and noise to -1
        self.labels_ = np.full(self.n_samples, -2)
        
        cluster_id = 0
        
        for point_idx in range(self.n_samples):
            if self.labels_[point_idx] != -2:
                continue
                
            neighbors = self._region_query(X, point_idx)
            
            if len(neighbors) < self.min_samples:
                self.labels_[point_idx] = -1 # Mark as noise
            else:
                self._expand_cluster(X, point_idx, neighbors, cluster_id)
                cluster_id += 1
                
        return self.labels_
        
    def _expand_cluster(self, X, point_idx, neighbors, cluster_id):
        # Assign the initial point to the cluster
        self.labels_[point_idx] = cluster_id
        
        # Iterate through neighbors (can grow during iteration)
        i = 0
        while i < len(neighbors):
            neighbor_idx = neighbors[i]
            
            # If it was labeled noise, it's a border point. Assign it and continue.
            if self.labels_[neighbor_idx] == -1:
                self.labels_[neighbor_idx] = cluster_id
                
            # If it is unassigned, assign it and check its neighbors
            elif self.labels_[neighbor_idx] == -2:
                self.labels_[neighbor_idx] = cluster_id
                new_neighbors = self._region_query(X, neighbor_idx)
                
                if len(new_neighbors) >= self.min_samples:
                    neighbors = neighbors + new_neighbors
            
            i += 1
            
    def _region_query(self, X, point_idx):
        distances = np.linalg.norm(X - X[point_idx], axis=1)
        return np.where(distances <= self.eps)[0].tolist()


## 🧪 Data Generation and Training
We use a dataset consisting of two interleaving half circles, where K-Means would normally fail.

In [ ]:
X, y = make_moons(n_samples=400, noise=0.1, random_state=42)

dbscan = DBSCAN(eps=0.15, min_samples=6)
labels = dbscan.fit_predict(X)


## 📊 Visualization

In [ ]:
plt.figure(figsize=(8, 6))

unique_labels = set(labels)
colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]

for k, col in zip(unique_labels, colors):
    if k == -1:
        # Black used for noise.
        col = [0, 0, 0, 1]
        label_name = "Noise"
    else:
        label_name = f"Cluster {k}"

    class_member_mask = (labels == k)
    xy = X[class_member_mask]
    plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col), markeredgecolor='k', markersize=8, label=label_name)

plt.title('DBSCAN Clustering Result')
plt.legend()
plt.show()